# Customer Churn Analysis

**Portfolio Project | Python · Pandas · SQLite**

This notebook documents the development of a customer churn analysis that combines customer, subscription, and support data. The presentation has been structured for GitHub, recruiter review, and technical interview discussion while preserving the original analysis, code, outputs, and execution sequence.

> **Notebook status:** Customer-level exploratory analysis is in progress. Subscription, support, correlation, visualization, dashboard, and final recommendation sections remain planned.

> **Reproducibility note:** The notebook expects `customer_churn.db` to be available in the working directory. Cached outputs were cleared so GitHub can render the notebook reliably; run the notebook from top to bottom to regenerate tables and charts.

## Business Problem

Customer churn reduces recurring revenue and can indicate gaps in product value, pricing, subscription design, or customer support. This project prepares a unified analysis dataset and begins exploring which demographic and behavioral segments show different churn patterns.

### Project Objectives

1. Inspect and validate the available source tables.
2. Clean fields that do not contribute usable analytical information.
3. Combine customer, subscription, and support records.
4. Create analysis-ready churn and demographic features.
5. Explore customer distribution and churn across key segments.
6. Build toward validated insights and retention recommendations.

## Dataset Overview

The analysis uses a local SQLite database containing three source tables.

| Table | Analytical role | Key information |
| --- | --- | --- |
| `db_customer` | Customer profile | Customer ID, location, gender, date of birth, interests |
| `db_subscription` | Subscription and churn | Plan, contract, dates, charges, CLTV, churn score |
| `db_support` | Support experience | Complaints, escalations, CSAT, comments |

The tables are linked using `customerid`.

### Analysis Workflow

**Data loading** → **Validation** → **Cleaning** → **Master dataset** → **Feature engineering** → **Customer EDA** → **Insights and recommendations**

---

## Environment Setup

### Import Analysis Libraries

- **Objective:** Load the Python libraries used for data handling, visualization, numerical operations, and SQLite access.
- **Why this step matters:** Centralizing imports makes the notebook dependencies visible before any analysis begins.
- **Business relevance:** A reproducible environment allows another analyst to review or rerun the workflow.
- **Expected output:** No displayed result; the imported packages become available to later cells.

In [ ]:
# import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

---

## Data Loading

### Connect to the SQLite Database

- **Objective:** Open a connection to the local `customer_churn.db` database.
- **Why this step matters:** All source tables are stored in SQLite and must be accessed through an active connection.
- **Business relevance:** Reliable source access is required before customer, subscription, and support records can be analyzed.
- **Expected output:** A SQLite connection object confirming that the connection was created.

In [ ]:
conn = sqlite3.connect("customer_churn.db")

print(conn)

#### Business Interpretation

The displayed connection object confirms that the database connection was created successfully. No business conclusion is drawn at this stage.

### List Available Tables

- **Objective:** Query SQLite metadata to identify the tables available in the database.
- **Why this step matters:** Confirming table availability prevents assumptions about source names and scope.
- **Business relevance:** The table inventory defines which customer journey data can be included in the churn analysis.
- **Expected output:** A table listing the names of all available SQLite tables.

In [ ]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

tables = pd.read_sql(query, conn)

tables

#### Business Interpretation

The database provides three complementary sources: customer profiles, subscription behavior, and support interactions. Together, they support a broader view of churn.

### Inspect the Customer Table

- **Objective:** Load the customer table and inspect sample records, dimensions, and column names.
- **Why this step matters:** A structural review confirms the table grain and available demographic fields.
- **Business relevance:** Customer attributes support segmentation of churn by demographics and geography.
- **Expected output:** A data preview followed by the table shape and column index.

In [ ]:
customer = pd.read_sql("""
SELECT *
FROM db_customer;
""", conn)

customer.head()

In [ ]:
customer.shape

In [ ]:
customer.columns

#### Business Interpretation

The customer table provides the demographic and geographic attributes required for customer segmentation. Business conclusions are deferred until cleaning and merging are complete.

### Inspect the Subscription Table

- **Objective:** Load subscription records and review sample values, dimensions, and columns.
- **Why this step matters:** Subscription fields contain the plan, contract, pricing, cancellation, CLTV, and churn-score variables required for churn analysis.
- **Business relevance:** These attributes support revenue, plan, contract, value, and churn-risk comparisons.
- **Expected output:** A data preview followed by the table shape and column index.

In [ ]:
subscription= pd.read_sql(
    """
    Select * 
    from db_subscription;
    """, conn
)

subscription.head()

In [ ]:
subscription.shape

In [ ]:
subscription.columns

#### Business Interpretation

The subscription table contains the commercial and churn-related attributes needed for plan, contract, revenue, CLTV, and cancellation analysis.

### Support Table

#### Inspect the Support Table

- **Objective:** Load support interactions and review sample values, dimensions, and columns.
- **Why this step matters:** Support records may explain customer dissatisfaction through complaints, escalations, comments, and CSAT scores.
- **Business relevance:** Support experience can later be compared with churn behavior to identify service-related retention risks.
- **Expected output:** A data preview followed by the table shape and column index.

In [ ]:
support = pd.read_sql("""
SELECT *
FROM db_support;
""", conn)

support.head()

In [ ]:
support.shape

In [ ]:
support.columns

#### Business Interpretation

The support table introduces complaint, escalation, and satisfaction variables that can later be evaluated as potential churn drivers.

### Preserved Working Calculations

- **Objective:** Retain two standalone ratio calculations from the original learning workflow.
- **Why this step matters:** These calculations document the exploratory problem-solving process even though their downstream use is not yet specified.
- **Business relevance:** Preserving intermediate work demonstrates the progression of the analysis without rewriting beginner-stage code.
- **Expected output:** Two numeric ratio results.

> **Note:** These standalone calculations are preserved exactly as part of the original learning journey. Their analytical purpose should be documented if they are used in a later KPI or conclusion.

In [ ]:
(21/6)*100/100

In [ ]:
(6/21)*100

#### Business Interpretation

*(Add business insight here after the purpose of these working calculations is confirmed.)*

---

## Data Audit and Validation

This phase reloads the source tables and checks their structure, sample records, data types, and missing values before cleaning begins.

### Reload Source Tables for Validation

- **Objective:** Create fresh DataFrames for the customer, subscription, and support tables.
- **Why this step matters:** Reloading establishes a consistent starting point for the audit and cleaning stages.
- **Business relevance:** A controlled source state reduces the risk of validating partially transformed data.
- **Expected output:** Three refreshed DataFrames with no displayed output.

In [ ]:
customer = pd.read_sql("""
SELECT *
FROM db_customer
""", conn)

subscription = pd.read_sql("""
SELECT *
FROM db_subscription
""", conn)

support = pd.read_sql("""
SELECT *
FROM db_support
""", conn)

### Review Table Schemas and Data Types

- **Objective:** Inspect row counts, non-null counts, and data types for all source tables.
- **Why this step matters:** Schema inspection identifies incorrect data types and sparse fields before transformation.
- **Business relevance:** Accurate types and completeness checks are essential for reliable churn metrics and date-based features.
- **Expected output:** A schema summary for each DataFrame.

In [ ]:
customer.info()

In [ ]:

subscription.info()

In [ ]:
support.info()

#### Business Interpretation

The schema review identifies date fields that require conversion and sparse columns that require a cleaning decision before analysis.

### Preview Source Records

- **Objective:** Display the first rows of each source table.
- **Why this step matters:** Sample records provide a practical check of values, formats, keys, and category labels.
- **Business relevance:** Early inspection helps identify data issues that could distort customer segmentation or churn interpretation.
- **Expected output:** A five-row preview for each source DataFrame.

In [ ]:
customer.head()

In [ ]:
subscription.head()

In [ ]:
support.head()

#### Business Interpretation

The sample records confirm that the three tables share `customerid` and contain complementary customer, subscription, and support information.

### Assess Missing Values

- **Objective:** Count missing values in every column of the three source tables.
- **Why this step matters:** Missingness determines which fields are unusable, optional, or meaningful by design.
- **Business relevance:** Understanding missing data prevents incorrect customer counts and misleading aggregations.
- **Expected output:** Column-level null counts for each source table.

> **Note:** Missing cancellation fields can represent active customers, while completely empty fields may be candidates for removal. The existing code determines the specific cleaning actions below.

In [ ]:
customer.isnull().sum()

In [ ]:
subscription.isnull().sum()

In [ ]:
support.isnull().sum()

#### Business Interpretation

The null audit distinguishes between structurally empty fields and business-meaningful missing values, such as the absence of a cancellation date.

---

## Data Cleaning

The cleaning steps remove fields that contain no usable analytical information and convert date columns into datetime format for reliable time-based analysis.

### Remove Fully Unusable Columns

- **Objective:** Drop `pincode` from the customer table and `col_1` from the support table.
- **Why this step matters:** Both fields contain no populated values in the validation output.
- **Business relevance:** Removing empty columns keeps the analytical dataset focused and reduces visual clutter.
- **Expected output:** Updated customer and support DataFrames with the selected columns removed.

In [ ]:
# DROP useless columsn 
customer = customer.drop(columns=['pincode'])

support = support.drop(columns=['col_1'])

In [ ]:
customer.head()
support.head()

#### Business Interpretation

The displayed support preview confirms that the fully empty support column was removed. No business conclusion is drawn from this validation output.

### Convert Date Fields to Datetime

- **Objective:** Convert date-of-birth and subscription date fields from text-like values to Pandas datetime values.
- **Why this step matters:** Datetime types are required for age, tenure, renewal, and cancellation analysis.
- **Business relevance:** Reliable date handling supports lifecycle segmentation and churn-timing analysis.
- **Expected output:** Converted datetime columns with no displayed result.

> **Note:** The date-conversion code appears twice in the original notebook. Both cells are retained unchanged to preserve the learning journey and execution order.

In [ ]:
customer["dob"] = pd.to_datetime(customer["dob"])

subscription["subscription_start_date"] = pd.to_datetime(
    subscription["subscription_start_date"]
)

subscription["renewal_date"] = pd.to_datetime(
    subscription["renewal_date"]
)

subscription["cancellation_date"] = pd.to_datetime(
    subscription["cancellation_date"]
)

In [ ]:
customer["dob"] = pd.to_datetime(customer["dob"])

subscription["subscription_start_date"] = pd.to_datetime(
    subscription["subscription_start_date"]
)

subscription["renewal_date"] = pd.to_datetime(
    subscription["renewal_date"]
)

subscription["cancellation_date"] = pd.to_datetime(
    subscription["cancellation_date"]
)

### Inspect Converted Subscription Dates

- **Objective:** Display the subscription table after date conversion.
- **Why this step matters:** A visual check confirms that missing cancellation values are represented consistently and that date fields remain readable.
- **Business relevance:** Validated dates are needed before deriving status or lifecycle features.
- **Expected output:** The subscription DataFrame with converted date columns.

In [ ]:
subscription

### Validate Customer Data Types

- **Objective:** Confirm the customer schema after cleaning and date conversion.
- **Why this step matters:** This verifies that `dob` is stored as datetime and that the removed field is no longer present.
- **Business relevance:** A valid date-of-birth field is required for demographic feature engineering.
- **Expected output:** An updated customer schema summary.

In [ ]:
customer.info()

### Remove the Sparse Interests Column

- **Objective:** Drop the `interests` field from the customer table.
- **Why this step matters:** The missing-value audit shows that the field is populated for only a small portion of records.
- **Business relevance:** Removing a highly sparse field avoids overinterpreting an incomplete customer attribute.
- **Expected output:** An updated customer DataFrame without the `interests` column.

In [ ]:
# DROP useless columsn 
customer = customer.drop(columns=['interests'])

### Validate Subscription Data Types

- **Objective:** Confirm the subscription schema after date conversion.
- **Why this step matters:** The check verifies that start, renewal, and cancellation fields are stored as datetime.
- **Business relevance:** Correct temporal types support later churn, renewal, and tenure analysis.
- **Expected output:** An updated subscription schema summary.

In [ ]:
subscription.info()

#### Business Interpretation

The validated schemas confirm that the required date fields are ready for feature engineering and lifecycle analysis.

---

## Master Dataset Construction

The source tables are combined through the shared `customerid` key to create a single analysis-ready dataset.

### Merge Customer and Subscription Data

- **Objective:** Left-join customer profiles with subscription attributes using `customerid`.
- **Why this step matters:** Combining demographics and subscription behavior creates a broader customer view.
- **Business relevance:** The merged table supports churn comparisons across geography, demographics, plans, contracts, pricing, and value.
- **Expected output:** A `customer_subscription` DataFrame plus its resulting shape.

In [ ]:
customer_subscription = pd.merge(
    customer,
    subscription,
    on="customerid",
    how="left"
)

customer_subscription.shape

### Preview the Customer-Subscription Dataset

- **Objective:** Display the first rows of the newly merged dataset.
- **Why this step matters:** A sample preview confirms that columns from both source tables are aligned by customer ID.
- **Business relevance:** Correct joins are essential before adding support data or calculating churn features.
- **Expected output:** A five-row preview of `customer_subscription`.

In [ ]:
customer_subscription.head()

#### Business Interpretation

The customer and subscription tables merge into a 21-row dataset, establishing the base customer-subscription view used by later analysis.

### Add Support Data to the Master Dataset

The `customer_subscription` dataset contains customer profile and subscription information in one table.

The next step adds available customer-support interactions using the shared `customerid` key.

The merged result is stored in `final_df`, which becomes the primary dataset for feature engineering and exploratory analysis.

### Merge Support Interactions

- **Objective:** Left-join support records onto the customer-subscription dataset using `customerid`.
- **Why this step matters:** This adds service experience variables to the customer and subscription context.
- **Business relevance:** The combined dataset can later support analysis of CSAT, complaints, escalations, and churn.
- **Expected output:** A consolidated `final_df` containing customer, subscription, and available support fields.

In [ ]:
final_df = pd.merge(
    customer_subscription,
    support,
    on = "customerid",
    how = "left"
)

### Validate the Final Dataset Shape

- **Objective:** Check the dimensions of the fully merged dataset.
- **Why this step matters:** Comparing row counts before and after the support join reveals whether the join changes the dataset grain.
- **Business relevance:** Data grain affects every downstream customer count, percentage, and churn rate.
- **Expected output:** The number of rows and columns in `final_df`.

In [ ]:
final_df.shape

### Preview the Final Dataset

- **Objective:** Inspect the first rows and confirm that support fields were added.
- **Why this step matters:** A joined preview helps identify repeated customer IDs caused by multiple support interactions.
- **Business relevance:** Understanding repeated records is necessary when interpreting later counts as customer records versus unique customers.
- **Expected output:** A five-row preview of the consolidated dataset.

In [ ]:
final_df.head()

#### Business Interpretation

> **Data-grain note:** The support merge increases the dataset from 21 to 23 rows because at least one customer has multiple support records. Downstream uses of `count()` or `shape[0]` therefore describe joined records unless the analysis explicitly counts unique customer IDs. The original analysis is preserved unchanged.

---

## Feature Engineering

This section creates analysis-friendly features from the cleaned master dataset:

- **Age** from date of birth
- **Customer status** from cancellation-date availability
- **Age group** from defined demographic bands

Additional features such as tenure and risk categories remain part of the future project roadmap.

### Derived Column: Age

- **Purpose:** Calculate customer age from the year component of `dob`.
- **Business meaning:** Age represents the customer's current demographic stage at the time the notebook is run.
- **Why this feature is useful:** It supports age distribution analysis and age-based churn segmentation.
- **Expected output:** A numeric `age` column added to `final_df`.

> **Note:** Because the calculation uses `pd.Timestamp.today().year`, ages may change when the notebook is re-executed in a future year. Cached outputs are preserved in this portfolio version.

In [ ]:
final_df["age"] = (
    pd.Timestamp.today().year
    - final_df["dob"].dt.year
    )

### Validate the Age Feature

- **Objective:** Preview date of birth alongside the newly calculated age.
- **Why this step matters:** Side-by-side inspection checks whether the derived values are plausible.
- **Business relevance:** Reliable age values are required before customer age segmentation.
- **Expected output:** A preview containing `dob` and `age`.

In [ ]:
final_df[["dob", "age"]].head()

#### Business Interpretation

*(Add business insight here after the age distribution is evaluated.)*

> **Checkpoint:** The `age` feature has been created and previewed. The next step converts the cancellation field into a readable customer-status label.

### Derived Column: Customer Status

- **Purpose:** Create a readable churn indicator named `customer_status`.
- **Business meaning:** Customers without a cancellation date are labeled **Active**; customers with a cancellation date are labeled **churned**.
- **Why this feature is useful:** It supports churn KPIs, customer segmentation, and comparisons across demographic, subscription, and support attributes.
- **Expected output:** A two-category field derived from `cancellation_date`.

### Check Cancellation-Date Availability

- **Objective:** Identify which merged records have no cancellation date.
- **Why this step matters:** The null pattern is used directly to classify active and churned status.
- **Business relevance:** A readable status feature simplifies churn KPIs and segment comparisons.
- **Expected output:** A Boolean series where `True` indicates a missing cancellation date.

In [ ]:
final_df["cancellation_date"].isnull()

#### Business Interpretation

The Boolean output verifies which records will be classified as Active based on missing cancellation dates. No segment conclusion is drawn from this intermediate check.

### Prepare the Status Condition

- **Objective:** Reference the cancellation-date series through the `cancel` variable.
- **Why this step matters:** The variable is used as the condition source for the status assignment in the next cell.
- **Business relevance:** A clear condition supports consistent active-versus-churned classification.
- **Expected output:** A reusable Series reference with no displayed result.

In [ ]:
#importing library 
import numpy as np

cancel = final_df["cancellation_date"]

### Create the Customer-Status Feature

- **Objective:** Assign `Active` when the cancellation date is missing and `churned` otherwise.
- **Why this step matters:** Raw dates are less convenient than a categorical status for grouping and KPI calculations.
- **Business relevance:** The feature becomes the primary churn label used throughout the notebook.
- **Expected output:** A new `customer_status` column in `final_df`.

In [ ]:
final_df["customer_status"] = np.where(
        cancel.isnull(),
        "Active",
        "churned"
)

### Validate Customer Status

- **Objective:** Preview cancellation date and customer status together.
- **Why this step matters:** Side-by-side inspection confirms that the conditional labels match the source field.
- **Business relevance:** A trustworthy churn label is required before any retention comparison.
- **Expected output:** A ten-row preview of `cancellation_date` and `customer_status`.

In [ ]:
final_df[["cancellation_date", "customer_status"]].head(10)

#### Business Interpretation

The preview confirms that missing cancellation dates map to `Active` and populated cancellation dates map to `churned`.

---

## Initial Descriptive Checks

### Profile Customer Records by Country

- **Objective:** Group the merged dataset by country and inspect customer-ID frequencies.
- **Why this step matters:** This is an initial geographic distribution check using the current joined dataset.
- **Business relevance:** Country concentration can guide market-development and localization questions.
- **Expected output:** Customer-ID frequencies within each country.

In [ ]:
final_df.groupby("country")["customerid"].value_counts()
#our subscription is mainly used by indian people with 19 subscribers and there is one person from nepal who binge our subscription

#### Business Interpretation

The original exploratory interpretation is retained in the code cell above. A final portfolio insight should be added after validating the intended customer-count denominator.

### Count Customer Records by State

- **Objective:** Aggregate the merged dataset by state.
- **Why this step matters:** State-level counts provide a more detailed geographic view than country totals.
- **Business relevance:** Regional distribution can support market prioritization and retention comparisons.
- **Expected output:** A customer-record count for each state.

In [ ]:
final_df.groupby("state")["customerid"].count()
# people are using our product across 9 different states with an average of 3 subscribers in each state

#### Business Interpretation

The original exploratory interpretation is retained in the code cell above. A final geographic insight can be added after confirming whether the analysis should use joined records or unique customers.

### Count Active and Churned Records

- **Objective:** Aggregate records by `customer_status`.
- **Why this step matters:** This creates an initial view of the active-versus-churned mix.
- **Business relevance:** Status counts are building blocks for churn KPIs and retention monitoring.
- **Expected output:** Record counts for Active and churned categories.

In [ ]:
final_df.groupby("customer_status")["customerid"].count()
# we have still 15 active customers and 8 people have cancelled our subscription 

#### Business Interpretation

The original active-versus-churned interpretation is retained in the code cell above. Final KPI wording should reflect the joined dataset grain.

### Count Records by Subscription Source

- **Objective:** Aggregate records by `subscription_type`.
- **Why this step matters:** The distribution shows how customers entered the subscription base.
- **Business relevance:** Acquisition-source mix can later be compared with churn and customer value.
- **Expected output:** Record counts for each subscription type.

In [ ]:
final_df.groupby("subscription_type")["customerid"].count()
# we are growing organic which is more than paid and refferal on other hand paid and refferal are same with 7 subscribers

#### Business Interpretation

The original subscription-source interpretation is retained in the code cell above. A validated insight can be added after customer-count methodology is finalized.

### Count Records by Plan Type

- **Objective:** Aggregate records by `plan_type`.
- **Why this step matters:** Plan distribution establishes the relative size of each commercial tier.
- **Business relevance:** Plan mix can later be connected to churn, pricing, and CLTV.
- **Expected output:** Record counts for Basic, Premium, and Standard plans.

In [ ]:
final_df.groupby("plan_type")["customerid"].count()
# Mostl customers are using our standard plan with 9 subs where as basic plan have less subs 

#### Business Interpretation

The original plan-distribution interpretation is retained in the code cell above. A final business insight can be added after the population definition is validated.

### Check the Final Row Count

- **Objective:** Return the number of rows in the joined analysis dataset.
- **Why this step matters:** This confirms the current record count after all merges.
- **Business relevance:** The denominator used in later calculations must be interpreted in the context of the joined data grain.
- **Expected output:** A single row-count value.

In [ ]:
len(final_df)

### Review Status Frequencies

- **Objective:** Return value counts for the customer-status feature.
- **Why this step matters:** This verifies the distribution of the newly created churn label.
- **Business relevance:** Status frequencies support active-customer and churned-customer KPI development.
- **Expected output:** Counts for Active and churned records.

In [ ]:
final_df["customer_status"].value_counts()

#### Business Interpretation

These outputs establish the current joined-record total and its Active/churned split. The original calculation logic is preserved.

### KPI Framework

The following KPI framework outlines the business questions that the notebook is preparing to answer.

| KPI                     | Business Question                                 |
| ----------------------- | ------------------------------------------------- |
| Total Customers         | How big is our customer base?                     |
| Active Customers        | How many paying customers do we currently have?   |
| Churned Customers       | How many have we lost?                            |
| Churn Rate %            | Are we retaining customers well?                  |
| Average Monthly Charges | How much does an average customer pay?            |
| Total Monthly Revenue   | How much recurring revenue are we generating?     |
| Average CLTV            | What is the average lifetime value of a customer? |
| Average Churn Score     | How risky is our customer base?                   |

### Inspect Fields Available for KPI Development

- **Objective:** Preview the master dataset before calculating headline KPIs.
- **Why this step matters:** The preview confirms that customer, status, charge, CLTV, churn score, and support fields are available.
- **Business relevance:** KPI calculations should be grounded in clearly identified source columns.
- **Expected output:** A two-row preview of `final_df`.

In [ ]:
final_df.head(2)

### Create the Active-Customer Subset

- **Objective:** Filter `final_df` to records labeled `Active` and display the subset size.
- **Why this step matters:** A dedicated active subset supports current-customer KPIs and follow-up analysis.
- **Business relevance:** Active-customer measures are central to retention and recurring-revenue reporting.
- **Expected output:** The number of records in `active_df`.

In [ ]:
# Total_Customers = final_df["customerid"].count()
active_df = final_df[final_df["customer_status"] == "Active"]
print(len(active_df))

### Alternate version 
# active_customers = len(
#     final_df[final_df["customer_status"] == "Active"]
# )

### Validate the Active Subset Shape

- **Objective:** Check the rows and columns in `active_df`.
- **Why this step matters:** The shape confirms that the filter retained the full dataset structure for active records.
- **Business relevance:** Validated subsets reduce the risk of using an incorrect KPI population.
- **Expected output:** The dimensions of `active_df`.

In [ ]:
active_df.shape

### Calculate the Total-Customer KPI

- **Objective:** Use the number of rows in `final_df` as the current total-customer measure.
- **Why this step matters:** This establishes a headline size metric using the notebook's existing logic.
- **Business relevance:** Total-customer volume is the denominator for several portfolio KPIs.
- **Expected output:** A single total-record value.

> **Important:** The support merge creates repeated rows for customers with multiple support interactions. The existing KPI code is preserved unchanged, so this measure reflects joined records rather than a distinct-customer count.

In [ ]:
# total customers KPI "Count the number of rows (customers)."
final_df.shape[0]

In [ ]:
final_df.shape[0]

#### Business Interpretation

The current KPI cells establish active-record and total-record measures. Revenue, CLTV, churn-rate, and churn-score KPIs remain to be completed.

---

## Exploratory Data Analysis (EDA)

### Customer Analysis

The current EDA focuses on customer demographics and churn status. Subscription, support, correlation, and visualization analyses are retained as planned extensions later in the notebook.

### Customer Analysis Questions

This section begins with the following business questions:

1. How many male and female customer records are present?
2. What percentage of customer records belongs to each gender?
3. How many active and churned records appear within each gender?
4. What is the churn rate within each gender group?

The analysis first calculates counts, then uses group-level denominators to calculate comparable churn rates.

### Preview the EDA Dataset

- **Objective:** Review sample rows before demographic analysis.
- **Why this step matters:** This confirms the active feature set and current row structure at the start of EDA.
- **Business relevance:** A final pre-analysis check supports trustworthy segment calculations.
- **Expected output:** A three-row preview of `final_df`.

In [ ]:
final_df.head(3)

#### Business Interpretation

The preview confirms that demographic, subscription, support, age, and status fields are available for the customer-analysis section.

### Standardize Gender Categories

- **Objective:** Replace `Men` with `Male` and `Women` with `Female`.
- **Why this step matters:** Equivalent labels must be standardized before grouping to avoid fragmented categories.
- **Business relevance:** Clean category labels produce accurate demographic counts and churn comparisons.
- **Expected output:** A normalized `gender` column with no displayed output.

> **Important:** Category standardization prevents equivalent labels from being counted as separate customer groups.

In [ ]:
# cleaning Gender column
final_df["gender"] = final_df["gender"].replace({
    "Men": "Male",
    "Women": "Female"
})

### Calculate Gender Distribution

- **Objective:** Count records by gender, retain the grouped totals, and calculate percentage shares.
- **Why this step matters:** Counts and percentages provide complementary views of demographic composition.
- **Business relevance:** Gender distribution establishes the population context needed for later churn-rate comparison.
- **Expected output:** Gender counts, the male-record count, and percentage distribution by gender.

In [ ]:
final_df.groupby("gender")["customerid"].count()

In [ ]:
total_gender = final_df.groupby("gender")["customerid"].count()

In [ ]:
final_df[final_df["gender"] == "Male"]["customerid"].count()

In [ ]:
total_gender

In [ ]:
Male_percentage = round((total_gender["Male"]/total_gender.sum()) * 100,2)
Male_percentage

In [ ]:
gender_percentage = round((total_gender / total_gender.sum()) * 100, 2)
gender_percentage

#### Business Interpretation

*(Add business insight here.)*

#### Age Distribution

### Summarize Customer Age

- **Objective:** Calculate descriptive statistics for the `age` feature.
- **Why this step matters:** Summary statistics show the center, spread, and range of the current age distribution.
- **Business relevance:** Age structure can inform customer segmentation and targeted retention strategies.
- **Expected output:** Count, mean, standard deviation, quartiles, minimum, and maximum age.

In [ ]:
final_df["age"].describe()

#### Business Interpretation

Notice something?

Youngest customer = 22
Oldest customer = 50
Average customer ≈ 37 years

There are no teenagers or elderly customers in this dataset.

### Derived Column: Initial Age Group

- **Purpose:** Create an initial two-category age segmentation using a cutoff of 35.
- **Business meaning:** Records below 35 are labeled `young adult`; all others are labeled `Adult`.
- **Why this feature is useful:** It demonstrates an initial segmentation approach before the categories are refined.
- **Expected output:** An `age_group` column and a preview of the updated dataset.

> **Learning journey:** This preliminary definition is intentionally preserved and is replaced by a more detailed three-band definition in the next step.

In [ ]:
final_df["age_group"] = np.where(
        final_df["age"] < 35,
        "young adult",
        "Adult"
)
final_df.head()

#### Business Interpretation

This two-group segmentation is an intermediate learning-stage feature and is refined below before distribution percentages are interpreted.

### Refine the Age-Group Definition

The initial two-group feature above is preserved as part of the learning journey. It is refined into three business-friendly age bands:

| Age range | Age group |
| --- | --- |
| 18–29 | Young Adult |
| 30–39 | Adult |
| 40+ | Senior Adult |

This segmentation provides more detail for demographic comparison than a two-group split.

### Derived Column: Refined Age Group

- **Purpose:** Replace the initial two-group split with three age bands.
- **Business meaning:** Ages 18–29 are `Young Adult`, ages 30–39 are `Adult`, and ages 40+ are `Senior Adult`.
- **Why this feature is useful:** More granular demographic bands support clearer comparison of customer composition and churn.
- **Expected output:** The updated `age_group` column and a preview of the revised categories.

In [ ]:
final_df["age_group"] = np.where(
    final_df["age"] >= 40,
    "Senior Adult",
    np.where(
        final_df["age"] >= 30,
        "Adult",
        "Young Adult"
)
)
final_df.head(2)

| Age   | Group        |
| ----- | ------------ |
| 18–29 | Young Adult  |
| 30–39 | Adult        |
| 40+   | Senior Adult |

### Calculate Age-Group Distribution

- **Objective:** Count customer records within each age group and convert the counts to percentages.
- **Why this step matters:** Percentages make groups comparable even when their record counts differ.
- **Business relevance:** The distribution shows which demographic bands are most represented in the current dataset.
- **Expected output:** A percentage share for each age group.

In [ ]:
# How many customers belong to each age group?
age_group_distribution = final_df.groupby("age_group")["customerid"].count()
# Adult and Senior adult have same customer distribution of 39%. where as young adult have lower of 21%

In [ ]:
age_percentage = round((age_group_distribution / age_group_distribution.sum()) * 100, 2)
age_percentage

#### Business Interpretation

The customer base is primarily composed of Adult and Senior Adult customers, with each group accounting for approximately 39% of the total customers. Young Adults represent only 21% of the customer base, indicating that the platform currently has lower representation among younger customers.

#### Country Distribution

### Calculate Country Distribution

- **Objective:** Count records by country and calculate each country's percentage share.
- **Why this step matters:** Both counts and percentages are needed to understand geographic concentration.
- **Business relevance:** Country distribution can guide market expansion and localization analysis.
- **Expected output:** Country-level counts and percentage shares.

In [ ]:
# Count customers in each country.
final_df.groupby("country")["customerid"].count()

In [ ]:
# Calculate the percentage for each country.
country_wise = final_df.groupby("country")["customerid"].count()

country_percent = round((country_wise/ country_wise.sum()) * 100, 2)
country_percent

#### Business Interpretation

The customer base is heavily concentrated in India, which accounts for approximately 95% of all customers. Nepal contributes only about 5% of the customer base, indicating limited market penetration outside India.

#### State Distribution

### Calculate State Distribution

- **Objective:** Count records by state and calculate each state's percentage share.
- **Why this step matters:** State-level analysis provides a more detailed geographic profile.
- **Business relevance:** Regional concentration can help prioritize local marketing, service, or retention analysis.
- **Expected output:** State-level counts and percentage shares.

In [ ]:
# Count customers by state.
final_df.groupby("state")["customerid"].count()

In [ ]:
# Calculate percentages.

customerby_state = final_df.groupby("state")["customerid"].count()

stateby_percent = round((customerby_state/ customerby_state.sum()) * 100, 2)
stateby_percent

#### Business Interpretation

Delhi has the highest customer representation, contributing approximately 17% of the total customer base. Karnataka, Maharashtra, Meghalaya, and Telangana each contribute around 13%, making them the next largest customer segments. The remaining states individually account for approximately 8% of customers, indicating a relatively dispersed customer distribution outside the top-performing states.

#### Active vs. Churn by Demographics

### Compare Status Counts by Gender

- **Objective:** Count Active and churned records within each gender.
- **Why this step matters:** The cross-tabulated counts show the composition of each gender-status combination.
- **Business relevance:** This is an intermediate step toward determining whether churn differs by gender.
- **Expected output:** Counts for each gender and customer-status combination.

In [ ]:
final_df.groupby(["gender","customer_status"])["customerid"].count()

#### Business Interpretation

❌ Your analysis is not complete yet.

This is where many junior analysts stop.

If I ask a manager:

"Are male customers churning more than female customers?"

Can you answer from this table?

No.

Why?

Because you're comparing counts, not rates.

For example:

Female: 4 churned out of 12
Male: 4 churned out of 11

The churn counts are the same (4), but the group sizes are different.

OUTPUT : 
| Gender | Active | Churned | Total |
| ------ | -----: | ------: | ----: |
| Female |      8 |       4 |    12 |
| Male   |      7 |       4 |    11 |

### Review Overall Gender-Status Percentages

- **Objective:** Convert the gender-status counts into percentages of all grouped records.
- **Why this step matters:** This calculation demonstrates that an overall denominator does not answer the within-gender churn-rate question.
- **Business relevance:** Recognizing denominator choice prevents misleading demographic comparisons.
- **Expected output:** Each gender-status combination as a percentage of the total grouped records.

In [ ]:
Churned_customers= final_df.groupby(["gender","customer_status"])["customerid"].count()
churn_percent = round((Churned_customers/ Churned_customers.sum()) * 100, 2)
churn_percent

#### Business Interpretation

The percentages above use the total grouped population as the denominator. The following steps retain the original learning progression and calculate churn within each gender instead.

##### Step 1: Count Churned Customers by Gender

- **Objective:** Isolate churned records and count them within each gender.
- **Why this step matters:** Churn-rate calculation requires a churned numerator for each group.
- **Business relevance:** Group-specific churn counts are the first input for fair retention comparisons.
- **Expected output:** Churned customer-record counts for Female and Male groups.

In [ ]:
churned_gender = final_df[
    final_df["customer_status"] == "churned"
].groupby("gender")["customerid"].count()

churned_gender

#### Business Interpretation

This output supplies the churned-record numerator for each gender. It is an intermediate calculation rather than a standalone conclusion.

##### Step 2: Count Total Customers by Gender

- **Objective:** Calculate the total record count within each gender.
- **Why this step matters:** Each gender needs its own denominator for a comparable churn rate.
- **Business relevance:** Using group-specific totals avoids misleading conclusions from raw churn counts.
- **Expected output:** Total customer-record counts for Female and Male groups.

In [ ]:
total_gender = final_df.groupby("gender")["customerid"].count()

total_gender

#### Business Interpretation

This output supplies the total-record denominator for each gender. It is combined with the preceding numerator in the next step.

##### Step 3: Calculate Churn Rate by Gender

- **Objective:** Divide churned records by total records within each gender.
- **Why this step matters:** Rates account for differences in group size.
- **Business relevance:** Comparable churn rates help identify whether one demographic segment may require greater retention attention.
- **Expected output:** A churn-rate percentage for each gender.

In [ ]:
churn_rate = round((churned_gender / total_gender) * 100, 2)

churn_rate

#### Business Interpretation

*(Add business insight here.)*

---

### Subscription Analysis

This section examines customer plan selection, contract duration, monthly charges, customer lifetime value, and churn patterns.

The `customer_subscription` dataset is used because it maintains one subscription record per customer and avoids duplicate rows introduced by the support-data merge.

In [ ]:
# — Plan distribution
plan_distribution = (
    customer_subscription
    .groupby("plan_type")["customerid"]
    .nunique()
    .sort_values(ascending=False)
)

plan_distribution

In [ ]:
# — Contract distribution
contract_distribution = (
    customer_subscription
    .groupby("contract_type")["customerid"]
    .nunique()
    .sort_values(ascending=False)
)

contract_distribution

In [ ]:
# — Average monthly charges by plan
avg_monthly_charges_by_plan = (
    customer_subscription
    .groupby("plan_type")["monthly_charges"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)

avg_monthly_charges_by_plan

In [ ]:
# — Average CLTV by plan
avg_cltv_by_plan = (
    customer_subscription
    .groupby("plan_type")["cltv"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)

avg_cltv_by_plan

In [ ]:
# — Average CLTV by contract
avg_cltv_by_contract = (
    customer_subscription
    .groupby("contract_type")["cltv"]
    .mean()
    .round(2)
    .sort_values(ascending=False)
)

avg_cltv_by_contract

In [ ]:
# — Create customer status
customer_subscription["customer_status"] = np.where(
    customer_subscription["cancellation_date"].isna(),
    "Active",
    "churned"
)

In [ ]:
# — Churn rate by plan

plan_customer_count = (
    customer_subscription
    .groupby("plan_type")["customerid"]
    .nunique()
)

churned_by_plan = (
    customer_subscription[
        customer_subscription["customer_status"] == "churned"
    ]
    .groupby("plan_type")["customerid"]
    .nunique()
)

churn_rate_by_plan = (
    churned_by_plan
    .div(plan_customer_count)
    .mul(100)
    .fillna(0)
    .round(2)
    .sort_values(ascending=False)
)

churn_rate_by_plan

### Subscription Analysis Summary

The subscription analysis compares customer volume, pricing, customer lifetime value, and churn across plans and contract types.

Because the dataset is small, these findings should be treated as directional observations rather than statistically representative conclusions.

### Support Analysis — Planned

Planned questions include:

- Average CSAT
- Escalations
- Complaints
- CSAT versus churn
- Escalations versus churn

> **Status:** Analysis code has not yet been added. Existing empty code cells are preserved unchanged.

In [ ]:
# Average CSAT
average_csat = support["csat_score"].mean().round(2)

average_csat

In [ ]:
# Total Escalations
total_escalations = (support["escalations"] == "Y").sum()

total_escalations

In [ ]:
# To see both Yes and No counts
support["escalations"].value_counts()

In [ ]:
# Merge support with subscription to get customer status

support_analysis = support.merge(
    customer_subscription[
        ["customerid", "customer_status"]
    ],
    on="customerid",
    how="left"
)

csat_by_status = (
    support_analysis
    .groupby("customer_status")["csat_score"]
    .mean()
    .round(2)
)

csat_by_status

In [ ]:
display(customer.head())
display(customer_subscription.head())
display(support.head())

## Visualization — Planned

Professional Matplotlib visualizations will be added to communicate the strongest validated findings.

> **Status:** Visualization code has not yet been added. Existing empty code cells are preserved unchanged.

#### Customer Status Distribution

In [ ]:
status_counts = final_df["customer_status"].value_counts()
status_counts

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))

status_counts.plot(kind="bar")

plt.title("Customer Status Distribution")
plt.xlabel("Customer Status")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)

plt.tight_layout()

plt.show()

### Insight

Most customers are currently Active, while approximately one-third have churned. This indicates that customer retention is generally healthy, but there is still an opportunity to reduce churn through targeted retention strategies.

In [ ]:
# Customer Distribution by Age Group

age_group_counts = (
    final_df["age_group"]
    .value_counts()
    .sort_index()
)

age_group_counts

In [ ]:
plt.figure(figsize=(8,4))

age_group_counts.plot(kind="bar")

plt.title("Customer Distribution by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Customers")

plt.xticks(rotation=0)

plt.tight_layout()

plt.show()

### Insight

The majority of customers belong to the 26–35 and 36–45 age groups. These segments represent the largest share of the customer base and may offer the greatest opportunity for customer retention and targeted marketing initiatives.

In [ ]:
# 

age_group_counts = (
    final_df["age_group"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(8,4))

ax = age_group_counts.plot(kind="bar")

plt.title("Customer Distribution by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Number of Customers")

plt.xticks(rotation=0)

# Add labels on top of each bar
for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()

plt.show()

In [ ]:
# Customer Distribution by State

In [ ]:
state_counts = (
    final_df["state"]
    .value_counts()
    .sort_values()
)

state_counts

In [ ]:
plt.figure(figsize=(8,6))

ax = state_counts.plot(kind="barh")

plt.title("Customer Distribution by State")
plt.xlabel("Number of Customers")
plt.ylabel("State")

# Add value labels 
for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()

plt.show()

### Insight

Customer distribution varies across states, with a few states contributing a larger share of the customer base. These regions could be prioritized for customer engagement and marketing initiatives.

In [ ]:
# Churn Rate by Gender

churn_by_gender = (
    final_df.groupby("gender")["customer_status"]
    .apply(lambda x : (x =="churned").mean() * 100)
    .round(2)
)

churn_by_gender

In [ ]:
plt.figure(figsize=(6,4))

ax = churn_by_gender.plot(kind="bar")

plt.title("Churn Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("Churn Rate (%)")

plt.xticks(rotation=0)

# Add labels 
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%")

plt.tight_layout()

plt.show()

### Insight

The churn rate is comparable across genders, indicating that gender alone is unlikely to be a strong driver of customer churn. Further analysis using factors such as subscription plan, contract type, or customer satisfaction may provide better insights into churn behavior.

## GitHub Portfolio — Planned

The intended portfolio package includes:

- Project README
- Business problem and dataset description
- Methodology
- Findings and recommendations
- Resume bullet points
- Technical interview discussion points

> **Status:** This documentation pass prepares the notebook for that final portfolio package.

---

## Executive Summary

This project analyzes customer churn by integrating customer, subscription, and support datasets stored in a SQLite database. The objective was to identify customer characteristics, subscription patterns, and support-related factors that may influence churn.

The analysis included data cleaning, feature engineering, exploratory data analysis (EDA), and visualization. Customer age, customer status, and age groups were derived to support deeper analysis. The three source tables were merged into a consolidated analytical dataset for reporting and insight generation.

Key analyses included customer demographics, churn distribution, subscription plan analysis, contract type analysis, customer support metrics, and churn rate comparisons across different customer segments. Visualizations were created using Pandas and Matplotlib to communicate the findings effectively.

Key Findings
* Approximately one-third of customers had churned, indicating an opportunity to improve customer retention.
* Most customers belonged to the 26–35 and 36–45 age groups.
* Customer distribution varied across states, with a few states contributing a larger share of the customer base.
* Churn rates were similar across genders, suggesting that gender alone is not a significant predictor of churn.
* Subscription plans, contract types, and customer support metrics are likely to provide stronger indicators of customer retention than demographic characteristics alone.